# Model Evaluation
This notebook imports reusable project code from `src/` and uses the real local M5 data when available.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))


In [ ]:
import pandas as pd
from src.config import settings
from src.data.loader import load_m5_data
from src.data.preprocessing import prepare_m5_long
prepared_path = settings.processed_dir / "m5_prepared.csv"
if prepared_path.exists():
    prepared = pd.read_csv(prepared_path, parse_dates=["date"])
else:
    data = load_m5_data()
    prepared = prepare_m5_long(data["sales"], data["calendar"], data["prices"])
print(prepared.shape)


In [ ]:
from src.forecasting.trainer import train_with_holdout
from src.forecasting.uncertainty import add_prediction_intervals
result = train_with_holdout(prepared, 28)
intervals = add_prediction_intervals(result["validation"], result["validation"]["residual"], 0.90)


In [ ]:
from src.evaluation.evaluator import evaluate_forecasts
evaluate_forecasts(result["validation"], history=result["train"])

In [ ]:
from src.forecasting.uncertainty import interval_coverage, average_interval_width
print("Coverage:", interval_coverage(intervals["demand"], intervals["lower_bound"], intervals["upper_bound"]))
print("Average width:", average_interval_width(intervals["lower_bound"], intervals["upper_bound"]))

In [ ]:
print("Compare the seasonal-naive and LightGBM backtest tables before claiming improvement.")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(result["validation"]["residual"], bins=30)
ax.set_title("Validation residual analysis")
ax.set_xlabel("Actual - forecast")
plt.show()


In [ ]:
from src.forecasting.backtesting import run_backtest
comparison, _ = run_backtest(prepared, horizon=28, n_folds=3)
comparison.groupby("model")[["MAE", "WMAPE", "RMSSE", "Bias", "interval_coverage"]].mean(numeric_only=True)
